# Fruit Freshness Detection(Fresh/Rotten) by Deep Learning(CNN)
## Dataset : Fruits fresh and rotten for classification( Kaggle )

## Importing Libraries and Dataset

In [20]:
import torch
import torch.nn as nn
import torch.optim as optim

import torchvision
from torch.utils.data import DataLoader
import torchvision.transforms as transforms
from torchvision import datasets

In [21]:
transform = transforms.Compose([
    transforms.Resize((32,32)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

In [22]:
trainset = datasets.ImageFolder(
    root=r"C:\Users\masud\anaconda_projects\april_26\deep_learning\Fruit_Freshness_detection/dataset\train",
    transform=transform
)

testset = datasets.ImageFolder(
    root=r"C:\Users\masud\anaconda_projects\april_26\deep_learning\Fruit_Freshness_detection/dataset\test",
    transform=transform
)

In [23]:
trainloader = DataLoader(trainset, batch_size = 16, shuffle = True)
testloader = DataLoader(testset, batch_size = 16)

## Define CNN model

In [24]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()

        self.conv_layers = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2), # kernel size = 2, stride = 2

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2) ,

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )

        self.fc_layers = nn.Sequential(
            nn.Linear(4*4*128, 256),
            nn.ReLU(),

            nn.Linear(256, 10)
        )

    def forward(self, x):
        x = self.conv_layers(x)
        x = x.view(x.size(0), -1) # flattening
        x = self.fc_layers(x)

        return x

In [25]:
model = CNN()

In [26]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

## Training the CNN

In [27]:
epochs = 10

for epoch in range(epochs):
    epoch_training_loss = 0.0

    for images, labels in trainloader:
        optimizer.zero_grad()
        
        output = model.forward(images) # FP
        loss = criterion(output, labels) # loss fnx
        loss.backward() # BP
        optimizer.step() # update params
        


        epoch_training_loss += loss.item()
        

    print(f"epoch={epoch+1}/{epochs} & loss={epoch_training_loss/len(trainloader)}")

epoch=1/10 & loss=0.6042647736196207
epoch=2/10 & loss=0.2374292343910893
epoch=3/10 & loss=0.17927255040707882
epoch=4/10 & loss=0.12307563544246929
epoch=5/10 & loss=0.09968535731015885
epoch=6/10 & loss=0.08507081473789374
epoch=7/10 & loss=0.06130933254227726
epoch=8/10 & loss=0.05068552335309174
epoch=9/10 & loss=0.05955915653065968
epoch=10/10 & loss=0.05397390058589532


## Model Evaluation

In [28]:
correct_labels = 0
total_labels = 0

model.eval()

with torch.no_grad():
    for images, labels in testloader:
        outputs = model.forward(images)
        _, predicted  = torch.max(outputs, 1)

        correct_labels += (predicted == labels).sum().item()
        total_labels += labels.size(0)

print(f"Test Accuracy = {correct_labels / total_labels * 100}")

Test Accuracy = 97.66493699036323


## Classification Metrics

In [29]:
y_true = []
y_pred = []

model.eval()

with torch.no_grad():
    for images, labels in testloader:
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)

        y_true.extend(labels.cpu().numpy())
        y_pred.extend(predicted.cpu().numpy())

from sklearn.metrics import classification_report

print(classification_report(y_true, y_pred))

              precision    recall  f1-score   support

           0       0.99      0.97      0.98       395
           1       0.98      1.00      0.99       381
           2       0.99      0.95      0.97       388
           3       0.97      0.97      0.97       601
           4       0.99      0.99      0.99       530
           5       0.94      0.98      0.96       403

    accuracy                           0.98      2698
   macro avg       0.98      0.98      0.98      2698
weighted avg       0.98      0.98      0.98      2698



## Determining the Overfitting

In [30]:
correct_labels = 0
total_labels = 0

model.eval()

with torch.no_grad():
    for images, labels in trainloader:
        outputs = model.forward(images)
        _, predicted  = torch.max(outputs, 1)

        correct_labels += (predicted == labels).sum().item()
        total_labels += labels.size(0)

print(f"Train Accuracy = {correct_labels / total_labels * 100}")

Train Accuracy = 98.98174479405559


## There is no overfitting
## Our model Accuracy is : 97.67%

In [32]:
torch.save(model.state_dict(), "model.pth")

In [33]:
model.load_state_dict(torch.load("model.pth"))
model.eval()

CNN(
  (conv_layers): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU()
    (8): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (fc_layers): Sequential(
    (0): Linear(in_features=2048, out_features=256, bias=True)
    (1): ReLU()
    (2): Linear(in_features=256, out_features=10, bias=True)
  )
)